In [1]:
# ============================================================
# CHANGE ONLY THIS CELL
# ============================================================

MODEL_FAMILY = "DA2"  # "DA2" or "DA3"

# One or more existing checkpoint files.
# Each checkpoint should already contain checkpoint["model_head"].
CHECKPOINT_PATHS = [
    "checkpoints/DA2_finetuned.pt",
    # "/path/to/another_checkpoint.pt",
]

# Your Hugging Face repo.
# Example: "myusername/depth-anything-custom-heads"
HF_REPO_ID = "ragerber13/depth-anything-custom-heads"

# Set True if you want the repo private.
HF_PRIVATE = False

# Optional:
# - None means use your existing `huggingface-cli login`, notebook login,
#   or HF_TOKEN environment variable.
# - Or paste a token here: "hf_..."
HF_TOKEN = '***REMOVED_HF_TOKEN***'

# Where checkpoint files should be stored inside the HF repo.
HF_CHECKPOINT_SUBFOLDER = "checkpoints"

# Base model IDs.
DA2_BASE_MODEL_ID = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf"
DA3_BASE_MODEL_ID = "depth-anything/da3metric-large"

# Device used for verification loading.
DEVICE = "cuda"  # "cuda", "cpu", or "mps"

# If True, strict load_state_dict is used for the head.
STRICT_LOAD = True

# If True, installs missing core packages.
# DA3 may still require the Depth-Anything-3 GitHub package depending on your environment.
INSTALL_MISSING_PACKAGES = True

# If True, verifies every checkpoint after upload.
VERIFY_AFTER_UPLOAD = True

In [2]:
# ============================================================
# DO NOT CHANGE BELOW THIS LINE
# ============================================================

import os
import sys
import json
import subprocess
from pathlib import Path
from typing import Dict, Any, List, Tuple

if INSTALL_MISSING_PACKAGES:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "torch",
        "torchvision",
        "transformers",
        "huggingface_hub",
    ])

import torch
from huggingface_hub import HfApi, create_repo, hf_hub_download


def normalize_device(device_str: str) -> torch.device:
    if device_str == "cuda" and not torch.cuda.is_available():
        print("CUDA requested but not available. Falling back to CPU.")
        return torch.device("cpu")

    if device_str == "mps":
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        print("MPS requested but not available. Falling back to CPU.")
        return torch.device("cpu")

    return torch.device(device_str)


device = normalize_device(DEVICE)


def get_hf_token():
    if HF_TOKEN is not None and HF_TOKEN.strip():
        return HF_TOKEN
    return os.environ.get("HF_TOKEN", None)


hf_token = get_hf_token()


def torch_load_checkpoint(path: str, map_location="cpu") -> Dict[str, Any]:
    """
    Loads your existing checkpoint format.

    First tries weights_only=True for safer tensor-only loading.
    Falls back to weights_only=False if your checkpoint contains objects that
    are not accepted by weights_only=True.

    Only use checkpoints you trust.
    """
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        # Older PyTorch versions do not support weights_only.
        return torch.load(path, map_location=map_location)
    except Exception as exc:
        print(f"weights_only=True failed for {path}. Retrying with weights_only=False.")
        print(f"Original error: {repr(exc)}")
        return torch.load(path, map_location=map_location, weights_only=False)


def get_head_module(model):
    if MODEL_FAMILY == "DA2":
        return model.head
    elif MODEL_FAMILY == "DA3":
        return model.model.head
    else:
        raise ValueError('Invalid MODEL_FAMILY. Accepted values are "DA2" or "DA3".')


def load_head_checkpoint(model, checkpoint_path: str):
    checkpoint = torch_load_checkpoint(checkpoint_path, map_location=device)

    if "model_head" not in checkpoint:
        raise KeyError(
            f'Checkpoint {checkpoint_path} does not contain key "model_head". '
            f"Available keys: {list(checkpoint.keys())}"
        )

    state_dict = checkpoint["model_head"]
    head = get_head_module(model)

    result = head.load_state_dict(state_dict, strict=STRICT_LOAD)

    print(f"Loaded checkpoint into model head: {checkpoint_path}")

    if not STRICT_LOAD:
        print("Missing keys:", result.missing_keys)
        print("Unexpected keys:", result.unexpected_keys)

    return result


def load_base_model():
    if MODEL_FAMILY == "DA2":
        from transformers import AutoModelForDepthEstimation

        model = AutoModelForDepthEstimation.from_pretrained(
            DA2_BASE_MODEL_ID
        )
        model = model.to(device)
        model.eval()
        return model

    elif MODEL_FAMILY == "DA3":
        from depth_anything_3.api import DepthAnything3

        model = DepthAnything3.from_pretrained(DA3_BASE_MODEL_ID)
        model = model.to(device)
        model.model.eval()
        return model

    else:
        raise ValueError('Invalid MODEL_FAMILY. Accepted values are "DA2" or "DA3".')


def make_repo_path(local_checkpoint_path: str) -> str:
    filename = Path(local_checkpoint_path).name

    if HF_CHECKPOINT_SUBFOLDER is None or HF_CHECKPOINT_SUBFOLDER.strip() == "":
        return filename

    return f"{HF_CHECKPOINT_SUBFOLDER.strip('/')}/{filename}"


def validate_checkpoint_before_upload(checkpoint_path: str):
    checkpoint_path = str(checkpoint_path)

    if not Path(checkpoint_path).exists():
        raise FileNotFoundError(f"Checkpoint does not exist: {checkpoint_path}")

    checkpoint = torch_load_checkpoint(checkpoint_path, map_location="cpu")

    if "model_head" not in checkpoint:
        raise KeyError(
            f'Checkpoint {checkpoint_path} does not contain key "model_head". '
            f"Available keys: {list(checkpoint.keys())}"
        )

    if not isinstance(checkpoint["model_head"], dict):
        raise TypeError(
            f'checkpoint["model_head"] should be a state_dict-like dict, '
            f"but got {type(checkpoint['model_head'])}"
        )

    print(f"Validated local checkpoint: {checkpoint_path}")


def upload_checkpoints() -> List[Dict[str, str]]:
    if not CHECKPOINT_PATHS:
        raise ValueError("CHECKPOINT_PATHS is empty.")

    for checkpoint_path in CHECKPOINT_PATHS:
        validate_checkpoint_before_upload(checkpoint_path)

    print(f"Creating or reusing Hugging Face repo: {HF_REPO_ID}")

    create_repo(
        repo_id=HF_REPO_ID,
        repo_type="model",
        private=HF_PRIVATE,
        exist_ok=True,
        token=hf_token,
    )

    api = HfApi(token=hf_token)

    uploaded = []

    for checkpoint_path in CHECKPOINT_PATHS:
        checkpoint_path = str(checkpoint_path)
        path_in_repo = make_repo_path(checkpoint_path)

        print(f"Uploading {checkpoint_path} -> {HF_REPO_ID}/{path_in_repo}")

        api.upload_file(
            path_or_fileobj=checkpoint_path,
            path_in_repo=path_in_repo,
            repo_id=HF_REPO_ID,
            repo_type="model",
            commit_message=f"Upload {Path(checkpoint_path).name}",
        )

        uploaded.append({
            "local_path": checkpoint_path,
            "path_in_repo": path_in_repo,
        })

    manifest = {
        "model_family": MODEL_FAMILY,
        "da2_base_model_id": DA2_BASE_MODEL_ID,
        "da3_base_model_id": DA3_BASE_MODEL_ID,
        "checkpoint_files": uploaded,
        "format": {
            "type": "torch_checkpoint",
            "required_key": "model_head",
            "description": "Checkpoint files are uploaded unchanged. Load base model first, then load checkpoint['model_head'] into the model head.",
        },
    }

    manifest_path = "hf_checkpoint_manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    api.upload_file(
        path_or_fileobj=manifest_path,
        path_in_repo="checkpoint_manifest.json",
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message="Upload checkpoint manifest",
    )

    print("Upload complete.")
    return uploaded


def compare_state_dicts(
    a: Dict[str, torch.Tensor],
    b: Dict[str, torch.Tensor],
    name_a: str,
    name_b: str,
) -> Tuple[bool, str]:
    keys_a = set(a.keys())
    keys_b = set(b.keys())

    if keys_a != keys_b:
        missing_in_b = sorted(keys_a - keys_b)
        missing_in_a = sorted(keys_b - keys_a)
        return False, (
            f"State dict keys differ.\n"
            f"Missing in {name_b}: {missing_in_b[:20]}\n"
            f"Missing in {name_a}: {missing_in_a[:20]}"
        )

    for key in sorted(keys_a):
        tensor_a = a[key].detach().cpu()
        tensor_b = b[key].detach().cpu()

        if tensor_a.shape != tensor_b.shape:
            return False, (
                f"Shape mismatch for key {key}: "
                f"{name_a} has {tuple(tensor_a.shape)}, "
                f"{name_b} has {tuple(tensor_b.shape)}"
            )

        if tensor_a.dtype != tensor_b.dtype:
            return False, (
                f"Dtype mismatch for key {key}: "
                f"{name_a} has {tensor_a.dtype}, "
                f"{name_b} has {tensor_b.dtype}"
            )

        if not torch.equal(tensor_a, tensor_b):
            max_abs_diff = (tensor_a - tensor_b).abs().max().item()
            return False, (
                f"Tensor values differ for key {key}. "
                f"Max absolute difference: {max_abs_diff}"
            )

    return True, "State dicts match exactly."


def verify_uploaded_checkpoint(model, local_path: str, path_in_repo: str):
    print("=" * 80)
    print(f"Verifying {HF_REPO_ID}/{path_in_repo}")

    downloaded_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=path_in_repo,
        repo_type="model",
        token=hf_token,
    )

    print(f"Downloaded to local cache: {downloaded_path}")

    local_checkpoint = torch_load_checkpoint(local_path, map_location="cpu")
    downloaded_checkpoint = torch_load_checkpoint(downloaded_path, map_location="cpu")

    local_head = local_checkpoint["model_head"]
    downloaded_head = downloaded_checkpoint["model_head"]

    same_file_content, message = compare_state_dicts(
        local_head,
        downloaded_head,
        "local checkpoint",
        "downloaded checkpoint",
    )

    if not same_file_content:
        raise AssertionError(
            f"Downloaded checkpoint does not match local checkpoint.\n{message}"
        )

    print("Local checkpoint and downloaded checkpoint match.")

    load_head_checkpoint(model, downloaded_path)

    loaded_head = {
        k: v.detach().cpu()
        for k, v in get_head_module(model).state_dict().items()
    }

    same_loaded_head, message = compare_state_dicts(
        downloaded_head,
        loaded_head,
        "downloaded checkpoint",
        "model head after load_state_dict",
    )

    if not same_loaded_head:
        raise AssertionError(
            f"Model head does not match downloaded checkpoint after loading.\n{message}"
        )

    print("Model head matches downloaded checkpoint after load_state_dict.")
    print(f"SUCCESS: {path_in_repo}")


uploaded_files = upload_checkpoints()

if VERIFY_AFTER_UPLOAD:
    print("Loading base model for verification...")
    model = load_base_model()

    for item in uploaded_files:
        verify_uploaded_checkpoint(
            model=model,
            local_path=item["local_path"],
            path_in_repo=item["path_in_repo"],
        )

    print("=" * 80)
    print("ALL CHECKPOINTS UPLOADED, DOWNLOADED, AND VERIFIED SUCCESSFULLY.")
else:
    print("Verification skipped because VERIFY_AFTER_UPLOAD=False.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 8.6 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.16.4
    Uninstalling huggingface_hub-1.16.4:
      Successfully uninstalled huggingface_hub-1.16.4
Validated local checkpoint: checkpoints/DA2_finetuned.pt
Creating or reusing Hugging Face repo: ragerber13/depth-anything-custom-heads
Uploading checkpoints/DA2_finetuned.pt -> ragerber13/depth-anything-custom-heads/checkpoints/DA2_finetuned.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete.
Loading base model for verification...


Loading weights:   0%|          | 0/503 [00:00<?, ?it/s]

Verifying ragerber13/depth-anything-custom-heads/checkpoints/DA2_finetuned.pt


checkpoints/DA2_finetuned.pt:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

Downloaded to local cache: /home/raphael/.cache/huggingface/hub/models--ragerber13--depth-anything-custom-heads/snapshots/42d86f6174decc51be6a7bff3e8f3a5e9b7eccbd/checkpoints/DA2_finetuned.pt
Local checkpoint and downloaded checkpoint match.
Loaded checkpoint into model head: /home/raphael/.cache/huggingface/hub/models--ragerber13--depth-anything-custom-heads/snapshots/42d86f6174decc51be6a7bff3e8f3a5e9b7eccbd/checkpoints/DA2_finetuned.pt
Model head matches downloaded checkpoint after load_state_dict.
SUCCESS: checkpoints/DA2_finetuned.pt
ALL CHECKPOINTS UPLOADED, DOWNLOADED, AND VERIFIED SUCCESSFULLY.
